### Zusammenhangskomponenten finden
Wir nutzen das Modul `matrix_helpers` und 
einen Algorithmus zu implementieren, der alle 
Felder in einem Gitter findet, 
die von einem Startfeld aus erreichbar sind.
Die erreichbaren Felder sind die Nachbarn.  
Um flexibel zu bleiben, benutzen wir eine Funktion, die uns
alle Nachbarn liefert.

Je nach Situation, interpretieren wir den Begriff Nachbar anders.
Hier einige Möglichkeiten:
- Alle gleichfarbigen Felder, die eine gemeinsame Seite haben.  
  Z.B. Felder finden, die ein Roboter erreichen kann, der sich in 4 Richtungen bewegen kann, Floodfill
- Alle gleichfarbigen Felder, die eine gemeinsame Ecke oder Seite haben.
  Z.B. Felder finden, die ein Roboter erreichen kann, der sich in 8 Richtungen bewegen kann.
- Wie oben, aber nur Felder einer bestimmten Farbe haben Nachbarn.  
  Z.B. beim Minesweeper alle sicheren Felder finden.


Die Implementation benutzt in typischer Weise einen Liste `stack`, die
alle Felder enthält, deren gleichfarbige Nachbarn wir noch nicht bestimmt haben.
Das Resultat ist eine Menge `component`.
Wir benutzen eine Menge, weil wir so schneller testen können, ob
ein Element bereits enthalten ist.

Der Algorithmus:
1. Wir erstellen eine Liste `stack = [start]` und eine Menge `component = {start}`.

2. Solange der `stack` nicht leer ist, machen wir Folgendes:
   - wir entfernen  das letzte Element.  
   - wir bestimmen alle Nachbarn dieses Feldes.
   - Ist ein Nachbar noch nicht in der Menge `component`, fügen wir dieses Feld hinzu.
     Zudem hängen wir das Feld an die Liste `stack` an, da wir
     von diesem Feld ja ev. noch weitere Felder erreichen können.

In [1]:
def get_component(matrix, start, get_neighbors):
    component = {start}
    stack = [start]
    while stack:
        pos0 = stack.pop()
        for pos in get_neighbors(matrix, pos0):
            if pos not in component:
                component.add(pos)
                stack.append(pos)
    return component

In [2]:
import random
import matrix_helpers as M


N = 10
matrix = M.make_matrix(N)
for i in range(N*N):
    M.set_item(matrix, M.idx2pos(i, ncol=N), random.randint(0, 2))
M.show_matrix(matrix)

0,2,0,1,1,2,0,1,2,1
1,2,2,2,2,2,0,0,2,2
2,0,1,0,0,2,0,1,1,0
1,1,2,1,0,1,0,1,2,0
2,0,1,1,0,2,2,1,2,0
0,1,1,1,2,2,1,0,0,0
1,2,2,0,2,2,0,1,0,0
1,2,2,2,0,1,1,0,0,0
0,2,2,0,0,0,1,2,0,0
0,1,1,1,2,0,0,1,2,2


In [16]:
def get_samecolored_8neighbors(matrix, pos0):
    ns = M.get_neighbors(matrix, pos0, kinds='sc')
    return [pos for pos in ns
            if M.get_item(matrix, pos) == M.get_item(matrix, pos0)
            ]


get_neighbors = get_samecolored_8neighbors

In [17]:
get_component(matrix, (0, 0), get_neighbors)

{(0, 0), (1, 0)}

In [18]:
def draw_matrix(canvas, matrix, grid_spec, cd):
    for row, cols in enumerate(matrix):
        for col, val in enumerate(cols):
            pos = (col, row)
            G.fill_rect(bg, pos, grid_spec, color=cd[val])

In [19]:
import widget_helpers as W
import grid_helpers as G


grid_spec = G.make_grid_spec(ncol=10)

mcanvas = W.get_mcanvas(3)
bg, mg, fg = mcanvas
mg.global_alpha = 0.4
mg.fill_style = 'white'

G.draw_grid(fg, grid_spec)
colordict = {0: 'red', 1: 'green', 2: 'blue'}
draw_matrix(bg, matrix, grid_spec, colordict)


mcanvas

KeyError: 9

In [7]:
def mark_component(matrix, pos):
    comp = get_component(matrix, pos, get_samecolored_8neighbors)
    mg.clear()
    for pos in comp:
        G.fill_rect(mg, pos, grid_spec)

In [8]:
mark_component(matrix, (0, 0))

In [9]:
def on_mouse_down(x, y):
    pos = G.xy2cr(x, y, grid_spec)
    mark_component(matrix, pos)

In [10]:
on_mouse_down(5, 25)

In [11]:
mcanvas.on_mouse_down(on_mouse_down)

In [12]:
matrix = [
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 1, 1, 1, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0, 1, 0, 0, 1, 0],
 [1, 0, 1, 1, 1, 1, 0, 0, 1, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 1, 0],
 [1, 0, 1, 0, 1, 1, 1, 1, 1, 0],
 [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
]


mg.clear()
bg.clear()
draw_matrix(bg, matrix, grid_spec, {0: 'grey', 1: 'blue'})
mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [13]:
def get_samecolored_4neighbors(matrix, pos0):
    ns = M.get_neighbors(matrix, pos0, kinds='s')
    return [pos for pos in ns
            if M.get_item(matrix, pos) == M.get_item(matrix, pos0)
            ]


def mark_component(matrix, pos):
    comp = get_component(matrix, pos, get_samecolored_4neighbors)
    mg.clear()
    for pos in comp:
        G.fill_rect(mg, pos, grid_spec)

In [14]:
matrix = [
 [0, 0, 1, 1, 1, 0, 0, 0, 1, 1],
 [1, 1, 2, 9, 2, 0, 0, 0, 1, 9],
 [9, 1, 2, 9, 3, 1, 1, 0, 1, 1],
 [1, 1, 1, 1, 2, 9, 1, 0, 0, 0],
 [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 2, 9, 2],
 [0, 0, 0, 0, 0, 1, 2, 4, 9, 2],
 [1, 1, 1, 0, 0, 1, 9, 9, 2, 1],
 [1, 9, 1, 0, 0, 1, 2, 2, 1, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
]


mg.clear()
bg.clear()
colordict = {0: 'grey', 1: 'green', 2: 'orange', 3: 'red', 4: 'blue', 9: 'black'}
draw_matrix(bg, matrix, grid_spec, colordict)
mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [15]:
def get_save_neighbors(matrix, pos0):
    if M.get_item(matrix, pos0) != 0:
        return []
    return M.get_neighbors(matrix, pos0, kinds='sc')


def mark_component(matrix, pos):
    comp = get_component(matrix, pos, get_save_neighbors)
    mg.clear()
    for pos in comp:
        G.fill_rect(mg, pos, grid_spec)